In [ ]:
from google.colab import userdata, drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests
import pandas as pd
import time

In [ ]:
API_BASE = "https://api.feriados.io"
API_KEY = userdata.get('API_Feriados')
PAIS = "EC"
ANIO_INICIAL = 2014
ANIO_FINAL = 2026

feriados = []
errores = []

headers = {
    "Authorization": f"Bearer {API_KEY}"
}

for anio in range(ANIO_INICIAL, ANIO_FINAL + 1):
    url = f"{API_BASE}/v1/{PAIS}/holidays/{anio}"

    try:
        respuesta = requests.get(
          url,
          headers=headers,
          timeout=30
        )

        respuesta.raise_for_status()
        resultado = respuesta.json()

        if resultado.get("success") is False:
            errores.append(anio)
            print(f"Error de la API para {anio}: {resultado}")
            continue

        datos = resultado.get("data", [])

        for feriado in datos:
            feriados.append({
                "fecha": feriado.get("date"),
                "nombre_feriado": feriado.get("name")
            })

        print(f"{anio}: {len(datos)} feriados obtenidos")

    except (requests.RequestException, ValueError) as error:
        errores.append(anio)
        print(f"Error consultando {anio}: {error}")

    time.sleep(0.2)

2014: 14 feriados obtenidos
2015: 12 feriados obtenidos
2016: 14 feriados obtenidos
2017: 12 feriados obtenidos
2018: 12 feriados obtenidos
2019: 14 feriados obtenidos
2020: 12 feriados obtenidos
2021: 15 feriados obtenidos
2022: 14 feriados obtenidos
2023: 12 feriados obtenidos
2024: 14 feriados obtenidos
2025: 14 feriados obtenidos
2026: 14 feriados obtenidos


In [ ]:
ruta_csv = "/content/drive/MyDrive/00_Proyecto_Sexto/Datos_Iniciales/feriados_ecuador_2014_2026.csv"

df_feriados = pd.DataFrame(
    feriados,
    columns=["fecha", "nombre_feriado"]
)

if not df_feriados.empty:
    df_feriados["fecha"] = pd.to_datetime(
        df_feriados["fecha"],
    )

    df_feriados = df_feriados.dropna(subset=["fecha"])

    df_feriados = (
        df_feriados
        .drop_duplicates(subset=["fecha", "nombre_feriado"])
        .sort_values("fecha")
        .reset_index(drop=True)
    )

    df_feriados.to_csv(
        ruta_csv,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y/%m/%d"
    )

    print(f"\nTotal de registros: {len(df_feriados)}")
    print("Archivo CSV generados correctamente.")
else:
    print("No se obtuvieron feriados. Revisa la URL y la respuesta de la API.")

if errores:
    print(f"Años que debes revisar: {errores}")


Total de registros: 173
Archivo CSV generados correctamente.
